# Order Reviews: BERT Pipeline
**Keyword Extraction (KeyBERT) + Sentiment Analysis (DistilBERT)**

Pipeline ini memproses kolom review_comment_message dari dataset order reviews dan menghasilkan:
- **keywords**,top keywords per review (KeyBERT + paraphrase-multilingual)
- **sentiment_label**, positive / negative / neutral
- **sentiment_positive/negative/neutral**, confidence score per kelas
- **sentiment_compound**, skor gabungan [-1, +1]

## Install Dependencies

In [1]:
# install semua library yang dibutuhkan
import subprocess, sys

packages = [
    "keybert",
    "sentence-transformers",
    "transformers",
    "torch",
    "pandas",
    "tqdm",
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

---
## Load Dataset

In [3]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

CSV_PATH = "order_reviews.csv"

df_raw = pd.read_csv(CSV_PATH)

print(f"Total baris  : {len(df_raw):,}")
print(f"Kolom        : {list(df_raw.columns)}")
df_raw.head(3)

Total baris  : 99,224
Kolom        : ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24


---
## Filter & Preprocessing

In [4]:
import re

def clean_text(text):
    text = str(text).strip()
    text = re.sub(r'[\r\n\t]+', ' ', text) # newline/tab/spasi
    text = re.sub(r'\s{2,}', ' ', text) # spasi ganda/tunggal
    text = re.sub(r'[^\w\s\u00C0-\u024F.,!?]', '', text)  # simpan karakter latin
    return text.strip()

# ambil hanya baris yang punya komentar
df = df_raw[df_raw['review_comment_message'].notna()].copy()
df['review_comment_message'] = df['review_comment_message'].apply(clean_text)

# buang komentar yang terlalu pendek
df = df[df['review_comment_message'].str.len() >= 5].reset_index(drop=True)

print(f"Baris dengan komentar valid: {len(df):,}")
print(f"Distribusi review_score:")
print(df['review_score'].value_counts().sort_index())

Baris dengan komentar valid: 40,028
Distribusi review_score:
review_score
1     8707
2     2125
3     3432
4     5710
5    20054
Name: count, dtype: int64


---
## Load Model KeyBERT & DistilBERT

**Model yang digunakan:**
- **KeyBERT** backbone: `paraphrase-multilingual-MiniLM-L12-v2`: ringan, mendukung 50+ bahasa termasuk Portugis
- **Sentiment**: `lxyuan/distilbert-base-multilingual-cased-sentiments-student` — DistilBERT fine-tuned multilingual sentiment

In [5]:
from keybert import KeyBERT
from transformers import pipeline as hf_pipeline

# KeyBERT dengan backbone multilingual
kw_model = KeyBERT(model='paraphrase-multilingual-MiniLM-L12-v2')

# DistilBERT Sentiment multilingual
sentiment_pipe = hf_pipeline(
    "text-classification",
    model="lxyuan/distilbert-base-multilingual-cased-sentiments-student",
    top_k=None,
    truncation=True,
    max_length=512,
    device=0
)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/373 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.92M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

---
## Sentiment Analysis (DistilBERT)

Proses dilakukan dalam **batch** untuk efisiensi.

In [6]:
from tqdm import tqdm
import math

BATCH_SIZE = 32

texts = df['review_comment_message'].tolist()
n_batches = math.ceil(len(texts) / BATCH_SIZE)

sentiment_labels   = []
sentiment_scores   = []
sentiment_positive = []
sentiment_negative = []
sentiment_neutral  = []

print(f"Memproses {len(texts):,} komentar dalam {n_batches} batch...")

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Sentiment"):
    batch   = texts[i : i + BATCH_SIZE]
    results = sentiment_pipe(batch)

    for res in results:
        # res = list of {label, score} untuk semua kelas
        label_map = {r['label']: r['score'] for r in res}
        pos = label_map.get('positive', 0.0)
        neg = label_map.get('negative', 0.0)
        neu = label_map.get('neutral',  0.0)

        # label dengan skor tertinggi
        best = max(res, key=lambda x: x['score'])

        sentiment_labels.append(best['label'])
        sentiment_scores.append(round(best['score'], 4))
        sentiment_positive.append(round(pos, 4))
        sentiment_negative.append(round(neg, 4))
        sentiment_neutral.append(round(neu, 4))

# tambahkan ke DataFrame
df['sentiment_label']    = sentiment_labels
df['sentiment_score']    = sentiment_scores
df['sentiment_positive'] = sentiment_positive
df['sentiment_negative'] = sentiment_negative
df['sentiment_neutral']  = sentiment_neutral

# compound score: positive - negative (range [-1, +1])
df['sentiment_compound'] = (
    df['sentiment_positive'] - df['sentiment_negative']
).round(4)

print("\nSentiment selesai!")
print(df['sentiment_label'].value_counts())

Memproses 40,028 komentar dalam 1251 batch...


Sentiment: 100%|██████████| 1251/1251 [03:40<00:00,  5.67it/s]


Sentiment selesai!
sentiment_label
positive    23499
negative    13853
neutral      2676
Name: count, dtype: int64


---
## Keyword Extraction (KeyBERT)

Menggunakan **MMR (Maximal Marginal Relevance)** untuk diversity keyword.

In [7]:
from tqdm import tqdm

TOP_N_KEYWORDS = 3

def extract_keywords(text: str, top_n: int = TOP_N_KEYWORDS) -> tuple[str, str]:

    if not isinstance(text, str) or len(text.strip()) < 5:
        return "", ""
    try:
        kws = kw_model.extract_keywords(
            text,
            keyphrase_ngram_range=(1, 2),   # unigram + bigram
            stop_words=None,                # model handle stopwords sendiri
            top_n=top_n,
            use_mmr=True,                   # diversity via MMR
            diversity=0.5,
        )
        if not kws:
            return "", ""
        keywords = " | ".join([k for k, _ in kws])
        scores   = " | ".join([str(round(s, 4)) for _, s in kws])
        return keywords, scores
    except Exception:
        return "", ""

print(f"Mengekstrak keyword dari {len(df):,} komentar...")

kw_keywords = []
kw_scores   = []

for text in tqdm(df['review_comment_message'], desc="KeyBERT"):
    kw, sc = extract_keywords(text)
    kw_keywords.append(kw)
    kw_scores.append(sc)

df['keywords']       = kw_keywords
df['keyword_scores'] = kw_scores

# top-1 keyword (untuk slicing mudah di Power BI)
df['top_keyword'] = df['keywords'].apply(
    lambda x: x.split(' | ')[0] if x else ""
)

print("\nContoh hasil:")
df[['review_comment_message', 'keywords', 'sentiment_label', 'sentiment_compound']].head(5)

Mengekstrak keyword dari 40,028 komentar...


KeyBERT: 100%|██████████| 40028/40028 [23:12<00:00, 28.74it/s]



Contoh hasil:


,review_comment_message,keywords,sentiment_label,sentiment_compound
0,Recebi bem antes do prazo estipulado.,do prazo | bem antes | recebi,positive,0.0380
1,Parabéns lojas lannister adorei comprar pela I...,parabéns lojas | internet seguro | lannister a...,positive,0.7480
2,aparelho eficiente. no site a marca do aparelh...,como 3desinfector | do aparelho | marca correta,positive,0.2402
3,"Mas um pouco ,travando...pelo valor ta Boa.",um pouco | valor ta | mas,positive,0.6592
4,"Vendedor confiável, produto ok e entrega antes...",vendedor confiável | produto ok | entrega antes,positive,0.9039


---
## Feature Engineering Tambahan

Kolom-kolom ini mempermudah pembuatan visualisasi di Power BI.

In [8]:
# date features
df['review_creation_date']    = pd.to_datetime(df['review_creation_date'])
df['review_answer_timestamp'] = pd.to_datetime(df['review_answer_timestamp'])
df['review_year']      = df['review_creation_date'].dt.year
df['review_month']     = df['review_creation_date'].dt.month
df['review_month_name']= df['review_creation_date'].dt.strftime('%B')
df['review_quarter']   = df['review_creation_date'].dt.quarter.apply(lambda q: f'Q{q}')
df['review_yearmonth'] = df['review_creation_date'].dt.strftime('%Y-%m')

# comment length
df['comment_length'] = df['review_comment_message'].str.len()

def length_bucket(n):
    if n < 30:  return 'Short (<30 chars)'
    if n < 100: return 'Medium (30-100)'
    return 'Long (>100 chars)'

df['comment_length_bucket'] = df['comment_length'].apply(length_bucket)

# sentiment alignment dengan review_score
def sentiment_alignment(row):
    label = row['sentiment_label']
    score = row['review_score']
    if label == 'positive' and score >= 4: return 'Aligned'
    if label == 'negative' and score <= 2: return 'Aligned'
    if label == 'neutral'  and score == 3: return 'Aligned'
    return 'Misaligned'

df['sentiment_alignment'] = df.apply(sentiment_alignment, axis=1)

print(f"Total kolom final: {len(df.columns)}")
print(df.dtypes)

Total kolom final: 24
review_id                          object
order_id                           object
review_score                        int64
review_comment_title               object
review_comment_message             object
review_creation_date       datetime64[ns]
review_answer_timestamp    datetime64[ns]
sentiment_label                    object
sentiment_score                   float64
sentiment_positive                float64
sentiment_negative                float64
sentiment_neutral                 float64
sentiment_compound                float64
keywords                           object
keyword_scores                     object
top_keyword                        object
review_year                         int32
review_month                        int32
review_month_name                  object
review_quarter                     object
review_yearmonth                   object
comment_length                      int64
comment_length_bucket              object
sentiment_al

---
## Validasi & Quick Stats

In [10]:
print(f"Total baris diproses : {len(df):,}")

print("\nDistribusi Sentiment:")
print(df['sentiment_label'].value_counts().to_string())

print("\nRata-rata sentiment_compound per review_score:")
print(
    df.groupby('review_score')['sentiment_compound']
    .mean().round(4).to_string()
)

from collections import Counter

print("\nTop 10 keyword paling sering muncul:")
all_kw = ' | '.join(df['keywords'].dropna())
kw_list = [k.strip() for k in all_kw.split('|') if k.strip()]
top_kw = Counter(kw_list).most_common(10)
for kw, cnt in top_kw:
    print(f"   {kw:<30} {cnt:>5}x")

print(f"\nAlignment rate: "
      f"{df['sentiment_alignment'].value_counts(normalize=True)['Aligned']*100:.1f}% Aligned")

print("\nPreview 5 baris:")
df[[
    'review_score', 'sentiment_label',
    'sentiment_compound', 'top_keyword', 'review_yearmonth'
]].head()

Total baris diproses : 40,028

Distribusi Sentiment:
sentiment_label
positive    23499
negative    13853
neutral      2676

Rata-rata sentiment_compound per review_score:
review_score
1   -0.2362
2   -0.1642
3   -0.0505
4    0.2151
5    0.3885

Top 10 keyword paling sering muncul:
   do prazo                        2529x
   produto                         2147x
   muito bom                       1494x
   recebi produto                  1152x
   produto chegou                  1129x
   não recebi                      1117x
   muito                            972x
   bom                              949x
   chegou antes                     856x
   no prazo                         748x

Alignment rate: 66.3% Aligned

Preview 5 baris:


,review_score,sentiment_label,sentiment_compound,top_keyword,review_yearmonth
0,5,positive,0.0380,do prazo,2017-04
1,5,positive,0.7480,parabéns lojas,2018-03
2,4,positive,0.2402,como 3desinfector,2018-05
3,4,positive,0.6592,um pouco,2018-02
4,5,positive,0.9039,vendedor confiável,2018-05


---
## Export ke CSV (Power BI Ready)

CSV menggunakan encoding `utf-8-sig` agar karakter Portugis (é, ã, ç, dll.) terbaca dengan benar di Excel dan Power BI.

In [12]:
# Pilih kolom output untuk Power BI
OUTPUT_COLS = [
    # Identifiers
    'review_id',
    'order_id',

    # Original review data
    'review_score',
    'review_comment_title',
    'review_comment_message',
    'comment_length',
    'comment_length_bucket',

    # Sentiment (DistilBERT)
    'sentiment_label',
    'sentiment_score',
    'sentiment_positive',
    'sentiment_negative',
    'sentiment_neutral',
    'sentiment_compound',
    'sentiment_alignment',

    # Keywords (KeyBERT)
    'keywords',
    'keyword_scores',
    'top_keyword',

    # Date features
    'review_creation_date',
    'review_answer_timestamp',
    'review_year',
    'review_month',
    'review_month_name',
    'review_quarter',
    'review_yearmonth',
]

OUTPUT_PATH = "voc_fp_mci.csv"

df_out = df[OUTPUT_COLS].copy()
df_out.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')